In [5]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

In [6]:
NUTS2 = 'Thessaly'

In [7]:
base_folder = 'E:/Drive/NOA/MBD-Prediction/Grid Level/data'

In [8]:
for year_item in range(2024, 2025):
    data = pd.read_csv(f'{base_folder}/{NUTS2}/GR_{NUTS2}_Timeline_GRID_{year_item}_filled_mean.csv', encoding='utf-8')

    try:
        data.drop(columns=['Unnamed: 0'], inplace=True)
    except Exception:
        print("No column named \'Unnamed: 0\'")

    data['dt_placement'] = pd.to_datetime(data['dt_placement'])

    # Extract year, month, and day from the 'date' column
    data['year'] = data['dt_placement'].dt.year
    data['month'] = data['dt_placement'].dt.month
    data['day'] = data['dt_placement'].dt.day

    dataset_list = []
    groups = data.groupby(['x', 'y', 'year'])
    num_of_groups = len(groups)

    c = 0
    for group, frame in groups:
        print(f"Year: {year_item} | Iteration: {c}/{num_of_groups}", end='\r')
        temp_df = frame.copy()
        temp_df.reset_index(drop=True, inplace=True)
        temp_df['acc_rainfall_1week'] = temp_df['rainfall'].rolling(window=7, min_periods=0).sum()
        temp_df['acc_rainfall_2week'] = temp_df['rainfall'].rolling(window=14, min_periods=0).sum()
        temp_df['acc_rainfall_month'] = temp_df['rainfall'].rolling(window=30, min_periods=0).sum()
        temp_df['acc_rainfall_jan'] = temp_df['dt_placement'].apply(lambda x: temp_df.loc[temp_df['dt_placement'] <= x, 'rainfall'].sum())
        dataset_list.append(temp_df)
        c = c + 1

    dataset = pd.concat(dataset_list, ignore_index=True)
    dataset.reset_index(drop=True, inplace=True)

    dataset.to_csv(f'{base_folder}/{NUTS2}/GR_{NUTS2}_Timeline_GRID_{year_item}_filled_mean_rain.csv', encoding='utf-8')